Determine if reliance behaviors affect academic outcomes. Removing percentage of cautious use to prevent dummy variable trap.

In [4]:
import duckdb
import statsmodels.formula.api as smf
from pathlib import Path

query = """
    WITH student_reliance AS (
        SELECT 
            userId,
            COUNT(*) AS total_interactions,
            SUM(CASE WHEN reliance_category = 'THOUGHTLESS_USE' THEN 1 ELSE 0 END) AS thoughtless_count,
            SUM(CASE WHEN reliance_category = 'REFLECTIVE_USE' THEN 1 ELSE 0 END) AS reflective_count,
            SUM(CASE WHEN reliance_category = 'CAUTIOUS_USE' THEN 1 ELSE 0 END) AS cautious_count
        FROM read_parquet($dataset_path)
        GROUP BY userId
    ), grades_concatenated AS (
        SELECT *, 'f24' AS semester FROM read_csv($fall_sem)
        UNION ALL
        SELECT *, 's25' AS semester FROM read_csv($spring_sem)
    )
    SELECT 
        r.userId,
        r.total_interactions,
        (r.thoughtless_count * 100.0 / r.total_interactions) AS thoughtless_pct,
        (r.reflective_count * 100.0 / r.total_interactions) AS reflective_pct,
        (r.cautious_count * 100.0 / r.total_interactions) AS cautious_pct,
        
        g.llm AS assignment_avg,
        g.no_llm AS exam_avg,
        g.semester
    FROM student_reliance AS r
    JOIN grades_concatenated AS g 
      ON r.userId = g.userId
"""

with duckdb.connect() as connection:
    df_regression = connection.execute(query, {
        "dataset_path": str(Path("../data/outputs/clean_analytical_dataset.parquet")),
        "fall_sem": str(Path("/workspaces/StudyChat/scores/f24_grades_released_normalized.csv")),
        "spring_sem": str(Path("/workspaces/StudyChat/scores/s25_grades_released_normalized.csv"))
    }).df()

model = smf.ols('exam_avg ~ thoughtless_pct + reflective_pct + total_interactions + C(semester)', data=df_regression)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               exam_avg   R-squared:                       0.034
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     1.497
Date:                Tue, 07 Jul 2026   Prob (F-statistic):              0.205
Time:                        12:08:11   Log-Likelihood:                 173.63
No. Observations:                 175   AIC:                            -337.3
Df Residuals:                     170   BIC:                            -321.4
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.8504      0

Variance inflation factor testing to determine collinearity of reliance percentages. Used since the condition number is affected by the difference in variable scales.

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

variables = ['thoughtless_pct', 'reflective_pct', 'total_interactions']
X = df_regression[variables].dropna()

X = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF_Score"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

             Variable  VIF_Score
0               const  57.801242
1     thoughtless_pct   2.641874
2      reflective_pct   2.576705
3  total_interactions   1.053670
